# Create Qdrant Content Collection

## Init

### Imports

In [8]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import os
from dotenv import load_dotenv
import openai

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Clients

In [9]:

load_dotenv()
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

### Reusable functions

In [6]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

## Create collection

In [11]:
CONTENT_COLLECTION_NAME="Content-collection-00"

In [ ]:
# qdrant_client.delete_collection(CONTENT_COLLECTION_NAME)
# qdrant_client.create_collection(
#     collection_name=CONTENT_COLLECTION_NAME,
#     vectors_config={"text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)},
#     sparse_vectors_config={"bm25": SparseVectorParams(modifier=models.Modifier.IDF)}
# )

True

In [6]:
qdrant_client.create_payload_index(
    collection_name=CONTENT_COLLECTION_NAME,
    field_name="id",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [7]:
# define pydantic model for content payload
from pydantic import BaseModel
class Content(BaseModel):
    id: str
    content_text: str
    mediaType:str
    author:str

## Add medium articles

### Add first article, step by step

In [8]:
first_article = pd.read_json("content-files/01-the-guild-introduction.jsonl", lines=True)
first_article.head()


,article,section,text
0,The Guild — an introduction to a peer-run orga...,Overview of what The Guild is,The Guild is a peer-run organization where sof...
1,The Guild — an introduction to a peer-run orga...,Governance and structure,The organization should be flat and use member...
2,The Guild — an introduction to a peer-run orga...,Inspiration from historical guilds,Inspired by the artisan guilds of the past — w...
3,The Guild — an introduction to a peer-run orga...,Why the Guild matters,Decentralized technologies remind us that powe...
4,The Guild — an introduction to a peer-run orga...,The gap it fills,"Surprisingly, no such community exists yet at ..."


In [9]:
def preprocess_content_text(row):
    return f"Article title: {row['article']}\n Article section: {row['section']}\n Article content: {row['text']}"

In [10]:
# data_to_embed is a list of Content 
# preprocess_content_text for the content_text field
# id is a generated uuid
# mediaType is "article"
# author is "The Guild"
import uuid
data_to_embed = [Content(id=str(uuid.uuid4()), content_text=preprocess_content_text(row), mediaType="article", author="The Guild") for _, row in first_article.iterrows()]
data_to_embed[0]

Content(id='8eb797db-f90b-4870-afa8-daf6a6877795', content_text='Article title: The Guild — an introduction to a peer-run organization for developers\n Article section: Overview of what The Guild is\n Article content: The Guild is a peer-run organization where software developers certify each other’s skills, learn together, and create opportunities. It is built on the idea that developers are stronger when united.', mediaType='article', author='The Guild')

In [11]:
text_to_embed = [data.content_text for data in data_to_embed]

In [12]:
embeddings = get_embeddings_batch(text_to_embed)

In [13]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data.content_text,
                    model="qdrant/bm25"
                )
            },
            # convert data to dict
            payload=data.model_dump()
        )
    )
    i += 1

In [14]:
pointstructs

[PointStruct(id=1, vector={'text-embedding-3-small': [-0.04554327204823494, 0.022272316738963127, -0.011622338555753231, 0.013041459955275059, 0.004641708452254534, -0.015715451911091805, -0.05708020180463791, 0.06580516695976257, 0.001207238412462175, 0.0014716810546815395, 0.04930131509900093, -0.03800090774893761, 0.007417535409331322, -0.0240330770611763, 0.05618667975068092, 0.03011690266430378, -0.04803987592458725, 0.0004455284506548196, -0.013547349721193314, 0.034899864345788956, 0.015518351458013058, 0.0022173766046762466, -0.01503217127174139, 0.07610693573951721, -0.0011694608256220818, -0.022232895717024803, -0.023862257599830627, 0.01660897210240364, -0.014637970365583897, -0.0054629589430987835, 0.004539873450994492, -0.01567603088915348, 0.024821478873491287, 0.004526733420789242, -0.004148958250880241, 0.04286270961165428, 0.02587267942726612, -0.004352628253400326, 0.029722701758146286, -0.018711373209953308, -0.029302220791578293, -0.03476846590638161, -0.00776574574

In [15]:
len(pointstructs)

14

In [16]:
qdrant_client.upsert(
    collection_name=CONTENT_COLLECTION_NAME,
    points=pointstructs
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Reusable function

In [17]:
def process_and_upsert_article_content(df):
    data_to_embed = [Content(id=str(uuid.uuid4()), content_text=preprocess_content_text(row), mediaType="article", author="The Guild") for _, row in df.iterrows()]
    text_to_embed = [data.content_text for data in data_to_embed]
    embeddings = get_embeddings_batch(text_to_embed)
    pointstructs = []
    # get latest collection size
    collection_size = qdrant_client.count(collection_name=CONTENT_COLLECTION_NAME)
    i = collection_size.count + 1
    for embedding, data in zip(embeddings, data_to_embed):
        pointstructs.append(
            PointStruct(
                id=i,
                vector={
                    "text-embedding-3-small": embedding,
                    "bm25": Document(
                        text=data.content_text,
                        model="qdrant/bm25"
                    )
                },
                # convert data to dict
                payload=data.model_dump()
            )
        )
        i += 1
    qdrant_client.upsert(
        collection_name=CONTENT_COLLECTION_NAME,
        points=pointstructs
    )
    print(f"Upserted {len(pointstructs)} points for {df.shape[0]} rows")
    

### Add the other articles

In [18]:
second_article = pd.read_json("content-files/02-the-guild-plan.jsonl", lines=True)
process_and_upsert_article_content(second_article)

Upserted 25 points for 25 rows


In [19]:
third_article = pd.read_json("content-files/03-The Guild Genesis v0 is live build reputation together.jsonl", lines=True)
process_and_upsert_article_content(third_article)

Upserted 17 points for 17 rows


In [20]:
fourth_article = pd.read_json("content-files/04-The Guild — October Update Building Together.jsonl", lines=True)
print(len(fourth_article))
process_and_upsert_article_content(fourth_article)

17
Upserted 17 points for 17 rows


## Add Tweeter history

### Load tweeter history

In [3]:
import json
from pathlib import Path

# Reuse your existing model
from pydantic import BaseModel

class Content(BaseModel):
    id: str
    content_text: str
    mediaType: str
    author: str


# TWEETS_JS_PATH = "notebooks/personal-project/content-files/tweets.js"
TWEETS_JS_PATH = "./content-files/tweets.js"

def load_tweets_as_content(
    file_path: str = TWEETS_JS_PATH,
    author: str = "estienneantoin1",  # change to your handle if needed
) -> list[Content]:
    # Read the JS file
    raw = Path(file_path).read_text(encoding="utf-8")

    # Strip the JS prefix: `window.YTD.tweets.part0 = `
    prefix = "window.YTD.tweets.part0 = "
    if raw.startswith(prefix):
        raw_json = raw[len(prefix):].strip()
    else:
        # Fallback: split on first '=' in case the prefix is slightly different
        raw_json = raw.split("=", 1)[1].strip()

    # Remove trailing semicolon if present
    if raw_json.endswith(";"):
        raw_json = raw_json[:-1]

    # Parse JSON array
    data = json.loads(raw_json)

    # Convert to list[Content]
    contents: list[Content] = []
    for item in data:
        tweet = item.get("tweet", {})
        tweet_id = tweet.get("id_str") or str(tweet.get("id"))
        text = tweet.get("full_text") or ""

        contents.append(
            Content(
                id=tweet_id,
                content_text=text,
                mediaType="tweet",
                author=author,
            )
        )

    return contents

# Example usage in the notebook:
tweet_contents = load_tweets_as_content()
len(tweet_contents), tweet_contents[0]

(1160,
 Content(id='1994413250918613177', content_text='Agentic systems work best when they use minimal LLM.\nLLMs should handle only the nondeterministic steps — the rest should be hard logic.\nBut in early design, LLMs are great for exploring and finding the shortest path.\n#AI #Agents #AgenticAI #DevTools #LLMEngineering', mediaType='tweet', author='estienneantoin1'))

In [13]:
def process_and_upsert_tweeter_content(input: list[Content], batch_size: int = 100):
    """
    Upserts tweet Content objects to Qdrant in batches to avoid "payload too large" errors.

    Args:
        input (list[Content]): List of Content objects representing tweets.
        batch_size (int): Number of points to upsert in each batch.
    """
    # get latest collection size
    collection_size = qdrant_client.count(collection_name=CONTENT_COLLECTION_NAME)
    current_id = collection_size.count + 1

    total_points = 0
    for batch_start in range(0, len(input), batch_size):
        batch = input[batch_start : batch_start + batch_size]
        text_to_embed = [data.content_text for data in batch]
        embeddings = get_embeddings_batch(text_to_embed)
        pointstructs = []

        for embedding, data in zip(embeddings, batch):
            pointstructs.append(
                PointStruct(
                    id=current_id,
                    vector={
                        "text-embedding-3-small": embedding,
                        "bm25": Document(
                            text=data.content_text,
                            model="qdrant/bm25"
                        ),
                    },
                    payload=data.model_dump(),
                )
            )
            current_id += 1

        # Upsert this batch
        qdrant_client.upsert(
            collection_name=CONTENT_COLLECTION_NAME,
            points=pointstructs
        )
        total_points += len(pointstructs)
        print(f"Upserted batch {batch_start // batch_size + 1}: {len(pointstructs)} points")

    print(f"Upserted total {total_points} points for {len(input)} rows")

In [14]:
process_and_upsert_tweeter_content(tweet_contents)

Upserted batch 1: 100 points
Upserted batch 2: 100 points
Upserted batch 3: 100 points
Upserted batch 4: 100 points
Upserted batch 5: 100 points
Upserted batch 6: 100 points
Upserted batch 7: 100 points
Upserted batch 8: 100 points
Upserted batch 9: 100 points
Upserted batch 10: 100 points
Upserted batch 11: 100 points
Upserted batch 12: 60 points
Upserted total 1160 points for 1160 rows
